#### Correr la imagen y levantar contenedor en docker
- docker run -d --name mongo -p 27017:27017 mongo:8.0

#### Crear proyecto
- uv init --no-package intro_mongodb --python 3.12

#### Instalar dependencias
- uv add ipykernel pymongo

In [1]:
import pymongo

print(pymongo.__version__)

4.19.0


In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
client.admin.command("ping")

{'ok': 1.0}

In [ ]:
# Creando DB
db = client["escuela"]

In [ ]:
# Creando colección
alumnos = db["alumnos"]

### Create

In [5]:
# CRUD -> Create / Read / Update / Delete
# Insertando un documento a la colección
res = alumnos.insert_one(
    {
        "nombre": "Ana Torres",
        "edad": 21,
        "ciudad": "Lima",
        "cursos": ["Python", "SQL"]
    }
)

print(res.inserted_id)

6adab40db596571b84add7c8


In [6]:
alumnos.insert_many(
    [
        {
            "nombre": "Luis Ramos",
            "edad": 19,
            "ciudad": "Bogotá"
        },
        {
            "nombre": "María Paz",
            "edad": 23,
            "ciudad": "Lima",
            "beca": True
        }
    ]
)

InsertManyResult([ObjectId('6adab48fb596571b84add7c9'), ObjectId('6adab48fb596571b84add7ca')], acknowledged=True)

### Read

In [ ]:
# CRUD -> Create / Read / Update / Delete

# alumnos.find() -> SELECT * FROM alumnos
for alumno in alumnos.find():
    print(alumno)
    print("-" * 50)

{'_id': ObjectId('6adab40db596571b84add7c8'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
--------------------------------------------------
{'_id': ObjectId('6adab48fb596571b84add7c9'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}
--------------------------------------------------
{'_id': ObjectId('6adab48fb596571b84add7ca'), 'nombre': 'María Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
--------------------------------------------------


In [11]:
# alumnos.find({"ciudad": "Lima"}) -> SELECT * FROM alumnos WHERE ciudad = "Lima"
for alumno in alumnos.find({"ciudad": "Bogotá"}):
    print(alumno)
    print("-" * 50)

{'_id': ObjectId('6adab48fb596571b84add7c9'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}
--------------------------------------------------


In [17]:
# alumnos.find({"ciudad": "Lima"}) -> SELECT * FROM alumnos WHERE ciudad = "Lima" LIMIT 1
res = alumnos.find_one({"ciudad": "Lima"})
res
# for field in res:
#     print(f"key: {field} -> Valor: {res[field]}")

{'_id': ObjectId('6adab40db596571b84add7c8'),
 'nombre': 'Ana Torres',
 'edad': 21,
 'ciudad': 'Lima',
 'cursos': ['Python', 'SQL']}

In [19]:
# alumnos.find({}, {"nombre": 1, "edad": 1}) -> SELECT nombre, edad FROM alumnos
for alumno in alumnos.find({}, {"nombre": 1, "edad": 1, "_id": 0}):
    print(alumno)

{'nombre': 'Ana Torres', 'edad': 21}
{'nombre': 'Luis Ramos', 'edad': 19}
{'nombre': 'María Paz', 'edad': 23}


In [25]:
# alumnos.find().sort("edad": -1) -> SELECT * FROM alumnos ORDER BY edad DESC
for alumno in alumnos.find({}).sort({"edad": -1}):
    print(alumno)

{'_id': ObjectId('6adab48fb596571b84add7ca'), 'nombre': 'María Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6adab40db596571b84add7c8'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
{'_id': ObjectId('6adab48fb596571b84add7c9'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}


In [28]:
# alumnos.count_documents({}) -> SELECT COUNT(1) FROM alumnos
print(alumnos.count_documents({}))

# alumnos.count_documents({"edad": 21}) -> SELECT COUNT(1) FROM alumnos WHERE edad = 21
print(alumnos.count_documents({"ciudad": "Lima"}))

3
2


In [29]:
res = alumnos.insert_one(
    {
        "nombre": "Marisol Vásquez",
        "edad": 30,
        "direccion": {
            "ciudad": "Lima",
            "distrito": "Miraflores"
        },
        "cursos": ["Python", "MongoDB"]
    }
)

In [30]:
for alumno in alumnos.find({"direccion.distrito": "Miraflores"}):
    print(alumno)

{'_id': ObjectId('6adab899b596571b84add7cb'), 'nombre': 'Marisol Vásquez', 'edad': 30, 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'}, 'cursos': ['Python', 'MongoDB']}


In [32]:
# igual, no igual
# MongoDB -> $eq $ne
# SQL -> = / <> ó !=

for alumno in alumnos.find({
    "ciudad": {"$ne": "Lima"},
    "direccion.ciudad": {"$ne": "Lima"}
}):
    print(alumno)

{'_id': ObjectId('6adab48fb596571b84add7c9'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogotá'}


In [ ]:
# igual, no igual
# MongoDB -> $eq $ne
# SQL -> = / <> ó !=

# gt -> greater than (>)
# gte -> greater than equal (>=)
# lt -> less than (<)
# lte -> less than equal (<=)

for alumno in alumnos.find({
    "edad": {"$gt": 19}
}):
    print(alumno)

{'_id': ObjectId('6adab40db596571b84add7c8'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
{'_id': ObjectId('6adab48fb596571b84add7ca'), 'nombre': 'María Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6adab899b596571b84add7cb'), 'nombre': 'Marisol Vásquez', 'edad': 30, 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'}, 'cursos': ['Python', 'MongoDB']}


### Update

In [ ]:
alumnos.update_one(
    {"nombre": "Ana Torres"}, # filtro
    {"$set": {"ciudad": "Arequipa"}} # fija o establece
)

alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6adab40db596571b84add7c8'),
 'nombre': 'Ana Torres',
 'edad': 21,
 'ciudad': 'Arequipa',
 'cursos': ['Python', 'SQL']}

In [37]:
alumnos.update_many({}, {"$inc": {"edad": -1}})
alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6adab40db596571b84add7c8'),
 'nombre': 'Ana Torres',
 'edad': 20,
 'ciudad': 'Arequipa',
 'cursos': ['Python', 'SQL']}

In [39]:
alumnos.find_one({"nombre": "Marisol Vásquez"})

{'_id': ObjectId('6adab899b596571b84add7cb'),
 'nombre': 'Marisol Vásquez',
 'edad': 29,
 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'},
 'cursos': ['Python', 'MongoDB']}

In [ ]:
alumnos.update_one(
    {"nombre": "Ana Torres"}, # filtro
    {"$push": {"cursos": "MongoDB"}} # adicionar elementos a un arreglo
)

alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6adab40db596571b84add7c8'),
 'nombre': 'Ana Torres',
 'edad': 20,
 'ciudad': 'Arequipa',
 'cursos': ['Python', 'SQL', 'MongoDB']}

In [42]:
alumnos.update_one(
    {"nombre": "Ana Torres"}, # filtro
    {"$pull": {"cursos": "Python"}} # eliminar elementos a un arreglo
)

alumnos.find_one({"nombre": "Ana Torres"})

{'_id': ObjectId('6adab40db596571b84add7c8'),
 'nombre': 'Ana Torres',
 'edad': 20,
 'ciudad': 'Arequipa',
 'cursos': ['SQL', 'MongoDB']}

### Delete

In [43]:
list(alumnos.find())

[{'_id': ObjectId('6adab40db596571b84add7c8'),
  'nombre': 'Ana Torres',
  'edad': 20,
  'ciudad': 'Arequipa',
  'cursos': ['SQL', 'MongoDB']},
 {'_id': ObjectId('6adab48fb596571b84add7c9'),
  'nombre': 'Luis Ramos',
  'edad': 18,
  'ciudad': 'Bogotá'},
 {'_id': ObjectId('6adab48fb596571b84add7ca'),
  'nombre': 'María Paz',
  'edad': 22,
  'ciudad': 'Lima',
  'beca': True},
 {'_id': ObjectId('6adab899b596571b84add7cb'),
  'nombre': 'Marisol Vásquez',
  'edad': 29,
  'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'},
  'cursos': ['Python', 'MongoDB']}]

In [45]:
from pymongo.collection import ObjectId

alumnos.delete_one({"_id": ObjectId("6adab899b596571b84add7cb")})
list(alumnos.find())

[{'_id': ObjectId('6adab40db596571b84add7c8'),
  'nombre': 'Ana Torres',
  'edad': 20,
  'ciudad': 'Arequipa',
  'cursos': ['SQL', 'MongoDB']},
 {'_id': ObjectId('6adab48fb596571b84add7c9'),
  'nombre': 'Luis Ramos',
  'edad': 18,
  'ciudad': 'Bogotá'},
 {'_id': ObjectId('6adab48fb596571b84add7ca'),
  'nombre': 'María Paz',
  'edad': 22,
  'ciudad': 'Lima',
  'beca': True}]